In [1]:
from cProfile import label

import pandas as pd
from sklearn.model_selection import train_test_split
import torch
from torch.utils.data import Dataset, DataLoader
import torch.nn as nn
import torch.optim as optim
import matplotlib.pyplot as plt
from torch.utils.data.dataset import _T_co

In [2]:
torch.manual_seed(42)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

In [3]:
df =pd.read_csv('fmnist_small.csv')
df.head(10)

,label,pixel1,pixel2,pixel3,pixel4,pixel5,pixel6,pixel7,pixel8,pixel9,...,pixel775,pixel776,pixel777,pixel778,pixel779,pixel780,pixel781,pixel782,pixel783,pixel784
0,9,0,0,0,0,0,0,0,0,0,...,0,7,0,50,205,196,213,165,0,0
1,7,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
2,0,0,0,0,0,0,1,0,0,0,...,142,142,142,21,0,3,0,0,0,0
3,8,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
4,8,0,0,0,0,0,0,0,0,0,...,213,203,174,151,188,10,0,0,0,0
5,9,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
6,9,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
7,9,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
8,1,0,0,0,0,0,0,0,0,0,...,130,0,0,0,0,0,0,0,0,0
9,0,0,0,0,1,1,3,0,0,0,...,80,77,83,45,0,0,0,1,0,0


In [4]:
X = df.iloc[:,1:].values
y = df.iloc[:, 0].values

In [5]:
X_train ,X_test , y_train,y_test = train_test_split(X,y,test_size = 0.2,random_state = 42)

In [6]:
!pip install torchvision

In [10]:
# transformations
from torchvision.transforms import transforms
custom_transform = transforms.Compose([
    transforms.Resize(256),
    transforms.CenterCrop(224),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])


In [14]:
from PIL import Image
import numpy as np

class CustomDataset(Dataset):

    def __init__(self, features, labels, transform):
        self.features = features
        self.labels = labels
        self.transform = transform

    def __len__(self):
        return self.features.shape[0]

    def __getitem__(self, index):
        image = self.features[index].reshape(28,28)
        image = image.astype(np.uint8)
        image = np.stack([image]*3, axis=-1)
        image = Image.fromarray(image)
        image = self.transform(image)

        return  image , torch.tensor(self.labels[index], dtype=torch.long)

In [15]:
train_dataset = CustomDataset(X_train,y_train,custom_transform)
test_dataset = CustomDataset(X_test,y_test,custom_transform)

In [16]:
train_loader = DataLoader(train_dataset,batch_size = 32,shuffle = True, pin_memory = True)
test_loader = DataLoader(test_dataset,batch_size = 32,shuffle = False, pin_memory = True)

In [18]:
import torchvision.models as models

vgg16 = models.vgg16(pretrained=True)

/home/enky-yy/anaconda3/lib/python3.13/site-packages/torchvision/models/_utils.py:207: UserWarning: The parameter 'pretrained' is deprecated since 0.13 and may be removed in the future, please use 'weights' instead.
  warnings.warn(
/home/enky-yy/anaconda3/lib/python3.13/site-packages/torchvision/models/_utils.py:222: UserWarning: Arguments other than a weight enum or `None` for 'weights' are deprecated since 0.13 and may be removed in the future. The current behavior is equivalent to passing `weights=VGG16_Weights.IMAGENET1K_V1`. You can also use `weights=VGG16_Weights.DEFAULT` to get the most up-to-date weights.
  warnings.warn(msg)


Downloading: "https://download.pytorch.org/models/vgg16-397923af.pth" to /home/enky-yy/.cache/torch/hub/checkpoints/vgg16-397923af.pth


100%|██████████| 528M/528M [05:52<00:00, 1.57MB/s] 


In [19]:
for param in vgg16.features.parameters():
    param.requires_grad = False

In [20]:
vgg16.classifier = nn.Sequential(
    nn.Linear(25088, 1024),
    nn.ReLU(),
    nn.Dropout(0.5),
    nn.Linear(1024, 512),
    nn.ReLU(),
    nn.Dropout(0.5),
    nn.Linear(512, 256),
    nn.ReLU(),
    nn.Dropout(0.5),
    nn.Linear(256, 10),
)

In [21]:
vgg16 = vgg16.to(device)

In [22]:
learning_rate = 0.0001

In [23]:
epochs = 10

In [24]:
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(vgg16.parameters(), lr = learning_rate, weight_decay = 0.0001)

In [25]:
for epoch in range(epochs):

    total_epoch_loss = 0

    for batch_features , batch_labels in train_loader:
        batch_features = batch_features.to(device)
        batch_labels = batch_labels.to(device)

        output = vgg16(batch_features)
        loss = criterion(output, batch_labels)

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
        total_epoch_loss += loss.item()

    avg_epoch_loss = total_epoch_loss/len(train_loader)

In [25]:
# evaluation on test data
total = 0
correct = 0

with torch.no_grad():

  for batch_features, batch_labels in test_loader:

    # move data to gpu
    batch_features, batch_labels = batch_features.to(device), batch_labels.to(device)

    outputs = vgg16(batch_features)

    _, predicted = torch.max(outputs, 1)

    total = total + batch_labels.shape[0]

    correct = correct + (predicted == batch_labels).sum().item()

print(correct/total)

In [27]:
# evaluation on training data
total = 0
correct = 0

with torch.no_grad():

  for batch_features, batch_labels in train_loader:

    # move data to gpu
    batch_features, batch_labels = batch_features.to(device), batch_labels.to(device)

    outputs = vgg16(batch_features)

    _, predicted = torch.max(outputs, 1)

    total = total + batch_labels.shape[0]

    correct = correct + (predicted == batch_labels).sum().item()

print(correct/total)

0.99
